# Lighthouse v2 — Demo Notebook

This notebook walks through the full Lighthouse pipeline end-to-end:
**Load Data → Driver Analysis → Model Training & Validation → Forecasting → Account Reconciliation → Visualization**

### Prerequisites
1. Clone the repo and run `uv sync` from the repo root (see README for full installation guide)
2. Open this notebook in VS Code and select the `.venv` Python kernel

### Data
This demo uses sample data for a fictional HVAC company called Mighty Ducts, included in the `demo/` folder:
- `accounts.csv` — historical account-level financial data
- `drivers.csv` — external/internal driver time series
- `config.yml` — pipeline configuration (date ranges, methods, parameters)

Run cells sequentially from top to bottom.

## Setup & Imports

In [1]:
import time
from pathlib import Path

from lh_v2.account_reconciliation import apply_account_reconciliation
from lh_v2.driver_analysis import analyze_drivers_full
from lh_v2.forecasting import create_account_forecasts, train_and_validate_models
from lh_v2.forecasting.account_forecasting.model_forecasting.model_forecasting_types import (
    ModelForecastingInput,
)
from lh_v2.forecasting.account_forecasting.model_validation.model_training_types import (
    ModelTrainingInput,
)
from lh_v2.io.data_loading import load_data_driver_ranking

from lh_v2.io.plotting import plot_account_forecasts
from lh_v2.params import parse_yaml

%matplotlib inline

/Users/Matthew.J.Maitland/Projects/lighthouse-2026/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1 — Load Configuration

The YAML config defines everything the pipeline needs: which accounts and drivers to use,
historical / validation / forecast date ranges, and all hyper-parameters for driver analysis,
model training, and reconciliation.

In [2]:
config_pth = Path('config.yml')
lh_params = parse_yaml(config_pth)

print(f"Segment: {lh_params.segment}")
print(f"Region:  {lh_params.region}")
print(f"Accounts ({len(lh_params.accounts)}): {lh_params.accounts}")
print()
print(f"Training range:    {lh_params.general_params.training_start_date} → {lh_params.general_params.training_end_date}")
print(f"Validation range:  {lh_params.general_params.validation_start_date} → {lh_params.general_params.validation_end_date}")
print(f"Testing range:     {lh_params.general_params.testing_start_date} → {lh_params.general_params.testing_end_date}")

Segment: Residential
Region:  North America
Accounts (5): ['volume', 'net_revenue', 'cogs_total', 't_w_total', 'gross_margin']

Training range:    2019-01-01 → 2022-12-01
Validation range:  2023-01-01 → 2023-12-01
Testing range:     2024-01-01 → 2024-06-01


## Step 2 — Load Data

Ingest the account (target) and driver (feature) CSVs. The loader filters to the
configured accounts/drivers, aligns date indices, and returns an `AccountsDriversInfo`
object ready for analysis.

In [3]:
acc_pth = Path('accounts.csv')
driv_pth = Path('drivers.csv')

t0 = time.time()
dr_data = load_data_driver_ranking(
    lh_params=lh_params, account_source=acc_pth, driver_source=driv_pth
)
print(f"Data loaded in {time.time() - t0:.2f}s")

print(f"\nAccounts loaded: {list(dr_data.accounts.account_map.keys())}")
print(f"\nDrivers by classification:")
for classification in dr_data.classified_drivers.get_ordered_classifications():
    drivers = dr_data.classified_drivers.get_ordered_drivers(classification)
    print(f"  {classification} ({len(drivers)}): {drivers}")

Data loaded in 0.05s

Accounts loaded: ['volume', 'net_revenue', 'cogs_total', 't_w_total', 'gross_margin']

Drivers by classification:
  External (12): ['Average Electricity Price', 'CCI', 'Construction Spend - Commercial', 'Cooling Degree Days', 'Global Price of Energy', 'Heating Degree Days', 'Housing Starts: Total', 'Industrial Production: Total Index', 'Interest Rates: 90 Day', 'Metals Price - Aluminum Price', 'Natural Gas', 'PPI for HVAC']
  Internal (6): ['Satisfaction Score: Commercial Building Automation', 'Satisfaction Score: Commercial Industrial Solutions', 'Satisfaction Score: Commercial Large-scale HVAC Systems', 'Satisfaction Score: Residential Air Quality & Controls', 'Satisfaction Score: Residential Cooling Systems', 'Satisfaction Score: Residential Heating Systems']


## Step 3 — Driver Analysis

For each account, the driver analysis pipeline:
1. Computes correlations between each driver and the account
2. Removes collinear drivers (via VIF / PCA / hierarchical clustering)
3. Ranks remaining drivers
4. Selects the top-N drivers per classification (leading vs coincident)
5. Optimises lag periods for leading drivers

In [4]:
analysis_results = analyze_drivers_full(
    accounts_drivers_info=dr_data,
    general_params=lh_params.general_params,
    da_params=lh_params.driver_analysis_params,
)

print("\n--- Selected Drivers ---")
for account, classifications in analysis_results.selected_drivers.items():
    print(f"\n  {account}:")
    for classification, drivers in classifications.items():
        print(f"    {classification}: {drivers}")

print("\n--- Selected Lags ---")
formatted_lags = analysis_results.format_lags()
for account, driver_lags in formatted_lags.items():
    print(f"\n  {account}:")
    for driver, lag in driver_lags.items():
        print(f"    {driver}: lag={lag}")


--- Selected Drivers ---

  volume:
    External: ['Average Electricity Price', 'Natural Gas', 'Industrial Production: Total Index']
    Internal: ['Satisfaction Score: Residential Heating Systems', 'Satisfaction Score: Commercial Building Automation', 'Satisfaction Score: Residential Air Quality & Controls']

  net_revenue:
    External: ['Average Electricity Price', 'Natural Gas', 'Industrial Production: Total Index']
    Internal: ['Satisfaction Score: Residential Heating Systems', 'Satisfaction Score: Commercial Building Automation', 'Satisfaction Score: Residential Air Quality & Controls']

  cogs_total:
    External: ['Average Electricity Price', 'Natural Gas', 'Industrial Production: Total Index']
    Internal: ['Satisfaction Score: Residential Heating Systems', 'Satisfaction Score: Commercial Building Automation', 'Satisfaction Score: Residential Air Quality & Controls']

  t_w_total:
    External: ['Average Electricity Price', 'Natural Gas', 'Housing Starts: Total']
    Inter

## Step 4 — Model Training & Validation

Each account is trained with multiple forecasting methods (e.g. Linear Regression,
Random Forest, Holt-Winters Exponential Smoothing, ARIMAX). The best method is
selected per account based on RMSE% over the validation period.

In [5]:
training_input = ModelTrainingInput(
    accounts_drivers=dr_data.select_drivers(analysis_results.selected_drivers),
    lags=analysis_results.format_lags(),
    classifications=analysis_results.format_classifications(),
)

training_results = train_and_validate_models(
    model_training_info=training_input,
    general_params=lh_params.general_params,
    af_params=lh_params.account_forecast_params,
    df_params=lh_params.driver_forecast_params,
)

print("\n--- Best Forecasting Method per Account ---")
selected_methods = training_results.select_forecast_methods()
for account, method in selected_methods.items():
    print(f"  {account}: {method.value}")


--- Best Forecasting Method per Account ---
  volume: xgboost
  net_revenue: random_forest
  cogs_total: xgboost
  t_w_total: xgboost
  gross_margin: xgboost


### Validation Plots

Compare all forecasting methods against historical actuals over the validation period.
The selected (best) method is labelled.

In [ ]:
for account in training_results.account_map.keys():
    training_results.plot_all_forecasts(
        account=account,
        forecast_daterange=(
            lh_params.general_params.validation_start_date,
            lh_params.general_params.validation_end_date,
        ),
        historicals=dr_data.accounts[account],
    )

## Step 5 — Model Forecasting

Using the selected method and tuned hyper-parameters for each account, produce the
final out-of-sample forecasts over the configured forecast period.

In [ ]:
forecasting_input = ModelForecastingInput(
    accounts_drivers=dr_data.select_drivers(analysis_results.selected_drivers),
    lags=analysis_results.format_lags(),
    classifications=analysis_results.format_classifications(),
    selected_model=training_results.select_forecast_methods(),
    best_params=training_results.format_best_params(),
    validation_errors=training_results.get_selected_methods_errors(
        selected_methods=training_results.select_forecast_methods(),
        actuals=dr_data.accounts,
        val_date_range=(
            lh_params.general_params.validation_start_date,
            lh_params.general_params.validation_end_date,
        ),
    ),
)

forecasting_results = create_account_forecasts(
    forecasting_input=forecasting_input,
    general_params=lh_params.general_params,
    df_params=lh_params.driver_forecast_params,
)
print(f"Forecast date range: {forecasting_results.forecast_daterange}")

## Step 6 — Account Reconciliation

Independently-forecasted accounts may not satisfy accounting identities
(e.g. Revenue − COGS ≠ Gross Profit). The reconciliation step adjusts forecasts
so that configured formulas hold, using either algebraic enforcement or MINT.

In [ ]:
reconciliation_results = apply_account_reconciliation(
    forecasting_data=forecasting_results,
    reconciliation_params=lh_params.account_reconciliation_params,
)

### Forecast Plots — Pre vs Post Reconciliation

Overlay the raw and reconciled forecasts to see the reconciliation adjustments.

In [ ]:
for account in reconciliation_results.accounts_forecasts.get_ordered_accounts():
    print(f"Plotting forecast for {account}")
    plot_account_forecasts(
        accounts=[
            forecasting_results.accounts_forecasts[account],
            reconciliation_results.accounts_forecasts[account],
        ],
        forecast_daterange=forecasting_results.forecast_daterange,
        labels=["Pre-Reconciliation", "Post-Reconciliation"],
    )